In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import cv2 as cv
import numpy as np
import os
import time

In [ ]:
vid_path = input("Enter video path: ")
cap = cv.VideoCapture(vid_path)

width = int(cap.get(3))
height = int(cap.get(4))
fps = 25   # frames per second

In [ ]:
fourcc = cv.VideoWriter_fourcc(*'mp4v')
out = cv.VideoWriter("output.mp4")

In [ ]:
prev_time = time.time()
while True:

    ret, frame = cap.read()
    if not ret:
        break

    h, w = frame.shape[:2]

    # Converting to HSV
    hsv = cv.cvtColor(frame, cv.COLOR_BGR2HSV)

    # Detecting white lane
    lower_white = np.array([0,0,200])
    upper_white = np.array([180,30,255])
    white_mask = cv.inRange(hsv, lower_white, upper_white)

    # Detecting yellow lane
    lower_yellow = np.array([15,100,100])
    upper_yellow = np.array([35,255,255])
    yellow_mask = cv.inRange(hsv, lower_yellow, upper_yellow)

    mask = white_mask + yellow_mask

    # Region of interest (bottom half)
    roi = mask[int(h/2):h, :]

    y, x = np.where(roi > 0)

    vehicle_center = w // 2

    if len(x) > 200:

        # Quadratic Fit (to support Curved Lane Support)
        curve = np.polyfit(y, x, 2)

        y_vals = np.linspace(0, roi.shape[0], 50)
        x_vals = curve[0]*y_vals**2 + curve[1]*y_vals + curve[2]

        for i in range(len(y_vals)):
            cv.circle(frame,(int(x_vals[i]), int(y_vals[i] + h*0.5)),2,(0,255,0),-1)

        lane_center = int(np.mean(x))

        # Auto Lane Width Estimate
        lane_left = np.min(x)
        lane_right = np.max(x)
        lane_width = lane_right - lane_left
        # lane center
        cv.circle(frame,(lane_center, int(h*0.75)),8,(0,255,255),-1)

        # vehicle center
        cv.circle(frame,(vehicle_center, int(h*0.75)),8,(255,0,0),-1)

        offset = vehicle_center - lane_center
        offset_percent = (offset / w) * 100

        if abs(offset_percent) < 10:
            status = "SAFE"
            color = (0,255,0)

        elif abs(offset_percent) < 20:
            status = "CAUTION"
            color = (0,255,255)

        else:
            status = "UNSAFE"
            color = (0,0,255)
    else:

        status = "UNKNOWN"
        color = (0,0,255)
        offset_percent = 0

    # FPS Meter
    current_time = time.time()
    fps_value = 1 / (current_time - prev_time)
    prev_time = current_time

    # displaying texts
    cv.putText(frame,f"FPS: {int(fps_value)}",(40,140),cv.FONT_HERSHEY_SIMPLEX,0.8,(255,255,255),2) # dispaly of FPS metrer readings

    cv.putText(frame,status,(40,50),cv.FONT_HERSHEY_SIMPLEX,1,color,3) # to display real time status

    cv.putText(frame,f"Offset: {offset_percent:.2f} %",(40,90),cv.FONT_HERSHEY_SIMPLEX,0.8,(255,255,255),2) # to display the offset percentage

    out.write(frame)

    if cv.waitKey(1) == 27:
        break

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# dowloading the output video
os.mkdirs("frames", exist_ok=True)
from google.colab import files
files.download("output.mp4")

cap.release()
out.release()
cv.destroyAllWindows()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>